In [1]:
import pandas as pd
import requests
import time

utr_path = "../data/raw/TargetScan/UTR_Sequences.txt"

# Inspect column names
header = pd.read_csv(utr_path, sep="\t", nrows=0)
print(header.columns.tolist())

['Refseq ID', 'Gene ID', 'Gene Symbol', 'Species ID', 'UTR sequence']


In [2]:
columns = [
    "Refseq ID",
    "Gene ID",
    "Gene Symbol",
    "Species ID"
]

human_transcripts = []

for chunk in pd.read_csv(
    utr_path,
    sep="\t",
    usecols=columns,
    chunksize=100_000
):
    human_chunk = chunk[chunk["Species ID"] == 9606]
    human_transcripts.append(human_chunk)

utr_ids = pd.concat(human_transcripts, ignore_index=True)

# Remove duplicate transcript records
utr_ids = utr_ids.drop_duplicates(subset="Refseq ID")

print("Unique human transcripts:", len(utr_ids))
print("Unique genes:", utr_ids["Gene ID"].nunique())

display(utr_ids.head())

Unique human transcripts: 28352
Unique genes: 19474


,Refseq ID,Gene ID,Gene Symbol,Species ID
0,CDR1as,CDR1as,CDR1as,9606
1,ENST00000000233.5,ENSG00000004059.6,ARF5,9606
2,ENST00000000412.3,ENSG00000003056.3,M6PR,9606
3,ENST00000001008.4,ENSG00000004478.5,FKBP4,9606
4,ENST00000001146.2,ENSG00000003137.4,CYP26B1,9606


In [3]:
utr_ids["transcript_id_clean"] = (
    utr_ids["Refseq ID"]
    .str.replace(r"\.\d+$", "", regex=True)
)

transcript_ids = (
    utr_ids["transcript_id_clean"]
    .dropna()
    .unique()
    .tolist()
)

print("Transcripts to query:", len(transcript_ids))

Transcripts to query: 28352


In [19]:
# Checkpoint

checkpoint = pd.read_csv(
    "../data/processed/biotype_checkpoint.csv",
    index_col=0
)

biotype_results = checkpoint["biotype"].to_dict()

print("Recovered annotations:", len(biotype_results))

Recovered annotations: 25238


In [20]:
import requests
import time

url = "https://rest.ensembl.org/lookup/id"

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

batch_size = 50
max_retries = 3

# Only process transcripts not already annotated
remaining_ids = [
    tid for tid in transcript_ids
    if tid not in biotype_results
]

print("Already annotated:", len(biotype_results))
print("Remaining:", len(remaining_ids))

for i in range(0, len(remaining_ids), batch_size):

    batch = remaining_ids[i:i + batch_size]

    for attempt in range(max_retries):
        try:
            response = requests.post(
                url,
                headers=headers,
                json={"ids": batch},
                timeout=(10, 120)
            )

            response.raise_for_status()
            batch_results = response.json()

            for tid, annotation in batch_results.items():
                if annotation is not None:
                    biotype_results[tid] = annotation.get("biotype")

            break

        except requests.exceptions.RequestException as e:
            print(f"Batch failed (attempt {attempt + 1}): {e}")
            time.sleep(2 ** (attempt + 1))

    else:
        print("Stopping. Failed batch can be retried later.")
        break

    # Save checkpoint after every successful batch
    pd.DataFrame.from_dict(
        biotype_results,
        orient="index",
        columns=["biotype"]
    ).rename_axis("transcript_id_clean").to_csv(
        "../data/processed/biotype_checkpoint.csv"
    )

    if (i // batch_size + 1) % 10 == 0:
        print(f"Processed {min(i + batch_size, len(remaining_ids))} "
              f"of {len(remaining_ids)} remaining")

    time.sleep(0.2)

print("Total annotated:", len(biotype_results))

Already annotated: 25238
Remaining: 3114
Processed 500 of 3114 remaining
Processed 1000 of 3114 remaining
Processed 1500 of 3114 remaining
Batch failed (attempt 1): 503 Server Error: Service Unavailable for url: https://rest.ensembl.org/lookup/id
Processed 2000 of 3114 remaining
Processed 2500 of 3114 remaining
Processed 3000 of 3114 remaining
Total annotated: 25238


In [25]:
for tid, annotation in batch_results.items():
    if annotation is not None and annotation.get("biotype") is not None:
        biotype_results[tid] = annotation["biotype"]

print("Updated dictionary:", len(biotype_results))

Updated dictionary: 25238


In [26]:
# Save dictionary for later

pd.DataFrame.from_dict(
    biotype_results,
    orient="index",
    columns=["biotype"]
).rename_axis("transcript_id_clean").to_csv(
    "../data/processed/biotype_checkpoint.csv"
)

In [33]:
biotype_df = pd.DataFrame(
    biotype_results.items(),
    columns=["transcript_id_clean", "biotype"]
)

utr_annotated = utr_ids.merge(
    biotype_df,
    on="transcript_id_clean",
    how="left"
)

# Exclude unresolved biotypes from this first calculation
defined = utr_annotated.dropna(subset=["biotype"])

print("Transcripts with defined biotype:", len(defined))
print("\nBiotype distribution:")
display(
    defined["biotype"]
    .value_counts()
    .to_frame("count")
    .assign(
        proportion=lambda x: x["count"] / x["count"].sum() * 100
    )
    .round(2)
)

Transcripts with defined biotype: 25238

Biotype distribution:


,count,proportion
biotype,,
protein_coding,24307,96.31
lncRNA,493,1.95
retained_intron,164,0.65
nonsense_mediated_decay,136,0.54
protein_coding_CDS_not_defined,105,0.42
transcribed_unprocessed_pseudogene,9,0.04
unprocessed_pseudogene,5,0.02
protein_coding_LoF,5,0.02
transcribed_processed_pseudogene,4,0.02


In [34]:
unresolved = utr_annotated[
    utr_annotated["biotype"].isna()
]

print("Unresolved transcripts:", len(unresolved))
print(
    "Present in TargetScan UTR dataset:",
    unresolved["transcript_id_clean"].isin(
        utr_ids["transcript_id_clean"]
    ).sum()
)

Unresolved transcripts: 3114
Present in TargetScan UTR dataset: 3114


In [35]:
utr_annotated["biotype_assumed"] = (
    utr_annotated["biotype"].fillna("protein_coding_assumed")
)

# Group actual and assumed protein-coding transcripts together
utr_annotated["biotype_group"] = (
    utr_annotated["biotype_assumed"]
    .apply(
        lambda x: "protein_coding"
        if x in ["protein_coding", "protein_coding_assumed"]
        else "other"
    )
)

display(
    utr_annotated["biotype_group"]
    .value_counts()
    .to_frame("count")
    .assign(
        proportion=lambda x: x["count"] / x["count"].sum() * 100
    )
    .round(2)
)

,count,proportion
biotype_group,,
protein_coding,27421,96.72
other,931,3.28


In [36]:
# Identify protein-coding-related biotypes
coding_biotypes = [
    "protein_coding",
    "protein_coding_CDS_not_defined",
    "protein_coding_LoF"
]

utr_annotated["biotype_group"] = "other"

utr_annotated.loc[
    utr_annotated["biotype"].isin(coding_biotypes),
    "biotype_group"
] = "protein_coding"

# Provisionally classify unresolved transcripts as protein-coding
utr_annotated.loc[
    utr_annotated["biotype"].isna(),
    "biotype_group"
] = "protein_coding_assumed"

# Summarise
display(
    utr_annotated["biotype_group"]
    .value_counts()
    .to_frame("count")
    .assign(
        proportion=lambda x: x["count"] / len(utr_annotated) * 100
    )
    .round(2)
)

,count,proportion
biotype_group,,
protein_coding,24417,86.12
protein_coding_assumed,3114,10.98
other,821,2.90


In [47]:
utr_df = pd.read_csv(
    "../data/processed/human_3UTRs_clean.tsv.gz",
    sep="\t"
)

# Remove Ensembl version suffixes
utr_df["gene_id_clean"] = (
    utr_df["gene_id"].str.replace(r"\.\d+$", "", regex=True)
)

utr_df["transcript_id_clean"] = (
    utr_df["transcript_id"].str.replace(r"\.\d+$", "", regex=True)
)


print(utr_df.columns.tolist())

['transcript_id', 'gene_id', 'gene_symbol', 'utr_sequence', 'utr_length', 'gene_id_clean', 'transcript_id_clean']


In [48]:
# Merge biotype annotations with the full UTR dataset
gene_biotypes = utr_df[
    ["transcript_id_clean", "gene_id_clean", "gene_symbol"]
].merge(
    biotype_df,
    on="transcript_id_clean",
    how="left"
)

# Define protein-coding-related biotypes
coding_biotypes = [
    "protein_coding",
    "protein_coding_CDS_not_defined",
    "protein_coding_LoF"
]

# Classify each gene
def classify_gene(group):
    if group["biotype"].isin(coding_biotypes).any():
        return "confirmed_protein_coding"
    elif group["biotype"].isna().any():
        return "unresolved"
    else:
        return "other"

gene_biotype_status = (
    gene_biotypes
    .groupby("gene_id_clean")
    .apply(classify_gene)
    .rename("gene_biotype_status")
    .reset_index()
)

# Summarise
gene_summary = (
    gene_biotype_status["gene_biotype_status"]
    .value_counts()
    .to_frame("gene_count")
)

gene_summary["proportion"] = (
    gene_summary["gene_count"] /
    len(gene_biotype_status) * 100
)

display(gene_summary.round(2))

print("Total unique genes:", len(gene_biotype_status))

,gene_count,proportion
gene_biotype_status,,
confirmed_protein_coding,17669,90.74
unresolved,1326,6.81
other,478,2.45


Total unique genes: 19473


In [49]:
# Save transcript-level biotype annotations
biotype_df.to_csv(
    "../data/processed/transcript_biotypes.csv",
    index=False
)

# Save transcript-level UTR dataset with biotype annotations
utr_annotated.to_csv(
    "../data/processed/human_3UTRs_biotype_annotated.csv",
    index=False
)

# Save gene-level biotype classifications
gene_biotype_status.to_csv(
    "../data/processed/gene_biotype_status.csv",
    index=False
)

# Save gene-level summary
gene_summary.to_csv(
    "../data/processed/gene_biotype_summary.csv"
)

#### Transcript biotype validation

Biotype annotations were retrieved for 25,238 of 28,352 human TargetScan transcripts. The remaining 3,114 transcripts could not be resolved using the current Ensembl REST API but were retained in the analysis.

At gene level, the 19,473 unique genes were classified as:

| Classification | Genes | Proportion |
|---|---:|---:|
| Confirmed protein-coding | 17,669 | 90.74% |
| Unresolved | 1,326 | 6.81% |
| Other | 478 | 2.45% |

The dataset is predominantly protein-coding. The full human 3′UTR dataset will therefore be retained without biotype-based filtering.

**Limitation:** Unresolved transcript biotypes remain unconfirmed. The presence of a protein-coding transcript for a gene does not establish that the specific transcript containing a seed match was expressed in the experiment.